# TholScript in 60 seconds
Render a damaged inscription, watch the enhancement, read it back with the trained models.
Run from the project root: `jupyter notebook notebooks/demo.ipynb`

In [ ]:
import sys; sys.path.insert(0, '..')
import random
from matplotlib import pyplot as plt
from tholscript.config import DamageConfig
from tholscript.data.lexicon import sample_tamil_line
from tholscript.vision.renderer import render_canvas

rng = random.Random(7)
texts = [sample_tamil_line(rng).text for _ in range(3)]
canvas = render_canvas(texts, 'tamil', cfg=DamageConfig(level=2), rng=rng)
plt.figure(figsize=(12, 5))
plt.imshow(canvas.image, cmap='gray')
plt.title('synthetic slab photo (ground truth known)')
plt.axis('off')

In [ ]:
from tholscript.app.service import TranslateService
svc = TranslateService()
result = svc.translate_image(canvas.image, script='tamil')
for gt, line in zip(texts, result['lines']):
    print('GT  :', gt)
    print('READ:', line['corrected'])
    print('conf:', [round(c['fused'], 2) for c in line['chars']][:12], '...\n')

In [ ]:
import base64
import cv2
import numpy as np
buf = np.frombuffer(base64.b64decode(result['enhanced_b64']), dtype=np.uint8)
enh = cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE)
plt.figure(figsize=(12, 5))
plt.imshow(enh, cmap='gray')
plt.title('enhanced ink map (what the reader sees)')
plt.axis('off')